In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

# If needed, ensure the key columns match (e.g., both are strings)
hydrobasins['HYBAS_ID'] = hydrobasins['HYBAS_ID'].astype(str)
hydrobasin_totals = hydrobasin_totals.reset_index()  # ensure HYBAS_ID is a column
hydrobasin_totals['HYBAS_ID'] = hydrobasin_totals['HYBAS_ID'].astype(str)

In [ ]:
# Define the target folder path where your CSV files are stored
folder_path = os.path.join(base_path, "Jamaica-jupyter-notebooks", "Networks_by_catchment_damage_analysis")

In [ ]:
# Get a list of all CSV files in the folder using glob
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Print out the list of CSV file names (optional, for confirmation)
display("Found CSV files:", csv_files)

In [ ]:
# List to hold DataFrames
dataframes = []

# Read each CSV and add a 'subsector' column extracted from the filename
for file in csv_files:
    subsector = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file)
    df['subsector'] = subsector  # use the filename as the subsector
    dataframes.append(df)

# Combine all CSVs into one DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

# List of the columns you want totals for
cols_of_interest = [
    "EAD_undefended_amin_sum", 
    "EAD_undefended_mean_sum", 
    "EAD_undefended_amax_sum", 
    "EAEL_undefended_amin_sum", 
    "EAEL_undefended_mean_sum", 
    "EAEL_undefended_amax_sum"
]

# -------------------------------
# Option 1: Totals by Hydrobasin only
# -------------------------------

# Group by Hydrobasin (using the "Hybas Id" column) and sum the columns of interest
hydrobasin_totals = combined_df.groupby("HYBAS_ID")[cols_of_interest].sum()
display("Totals by Hydrobasin:")
display(hydrobasin_totals)

# Create an aggregated DataFrame for merging
aggregated_df = hydrobasin_totals.reset_index()
aggregated_df['HYBAS_ID'] = aggregated_df['HYBAS_ID'].astype(str)

# -------------------------------
# Option 2: Totals by Hydrobasin and Broad Infrastructure Group
# -------------------------------

# Define a mapping for grouping subsectors into broader categories
group_mapping = {
    "Transport": ["rail", "roads", "ports", "airports"],
    "Electricity": ["electricity"],
    "Water": ["wastewater", "irrigation", "pipelines"],
    "Buildings": ["buildings"]
}

# Function to map a subsector to its broader group
def map_subsector(subsector):
    subsector_lower = subsector.lower()
    for group, keywords in group_mapping.items():
        if any(keyword in subsector_lower for keyword in keywords):
            return group
    return "Other"

# Add a new column that holds the broad infrastructure group for each row
combined_df["grouped_subsector"] = combined_df["subsector"].apply(map_subsector)

# Group by both Hydrobasin and the grouped infrastructure category, then sum the key columns
hydrobasin_grouped_totals = combined_df.groupby(["HYBAS_ID", "grouped_subsector"])[cols_of_interest].sum()
display("\nTotals by Hydrobasin and Broad Infrastructure Subsector:")
display("\nAggregated Totals by Hydrobasin and Broad Infrastructure Group:")
display(hydrobasin_grouped_totals)

In [ ]:
# -------------------------------
# 3. Merge Aggregated Totals with Hydrobasins Geometry
# -------------------------------
merged_gdf = hydrobasins.merge(aggregated_df, on="HYBAS_ID", how="left")

# -------------------------------
# 4. Plotting the Aggregated Data for All Catchments
# -------------------------------
# Define common extents for your plot (adjust as needed)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)

fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the merged GeoDataFrame, using 'EAD_undefended_amax_sum' for color mapping.
# A colorbar is automatically added.
merged_gdf.plot(
    column="EAD_undefended_amax_sum", 
    ax=ax, 
    cmap="Reds", 
    edgecolor="black", 
    linewidth=0.5, 
    legend=True,
    legend_kwds={'label': "EAD Undefended Amax Sum", 'orientation': "vertical"}
)

# (Optional) If you have background layers, add them here:
# catchments_gdf.plot(ax=ax, color="lightgrey", edgecolor="white")
# jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# Set plot extents and labels
ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Catchments by Baseline EAD Undefended Max Value", 
             fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

plt.tight_layout()
plt.show()